In [1]:
import numpy as np
import pandas as pd
import datetime as dt
from salishsea_tools import evaltools as et
import datetime as dt
import matplotlib as mpl
import netCDF4 as nc

fs=16
mpl.rc('xtick', labelsize=fs)
mpl.rc('ytick', labelsize=fs)
mpl.rc('legend', fontsize=fs)
mpl.rc('axes', titlesize=fs)
mpl.rc('axes', labelsize=fs)
mpl.rc('figure', titlesize=fs)
mpl.rc('font', size=fs)
mpl.rc('font', family='sans-serif', weight='normal', style='normal')

%matplotlib inline

In [2]:
data = pd.read_csv('/ocean/atall/MOAD/Obs/PNW_obs_compiled.csv')
PATH= '/results2/SalishSea/nowcast-green.202111/'
data['time']=pd.to_datetime(data['time']) 

In [3]:
data = data.rename(columns={"latitude (degrees_north)": "Lat", "longitude (degrees_east)": "Lon", "time": "dtUTC", "depth (m)": "Z",
                                   "salinity (g kg-1)": "SA","temperature (degC)": "Temp","DO (umol kg-1)": "Oxygen_Dissolved",
                                   "NO3 (uM)": "NO3","NH4 (uM)": "NH4","time": "dtUTC"})
data['dtUTC']=pd.to_datetime(data.dtUTC).dt.tz_localize(None)

In [4]:
def matchData(
    data,
    filemap,
    fdict,
    mod_start=None,
    mod_end=None,
    mod_nam_fmt='nowcast',
    mod_basedir='/results2/SalishSea/nowcast-green.202111/',
    mod_flen=1,
    method='bin',
    meshPath=None,
    maskname='tmask',
    wrapSearch=False,
    fastSearch=False,
    wrapTol=1,
    e3tvar='e3t',
    fid=None,
    sdim=3,
    quiet=False,
    preIndexed=False
    ):
    """Given a discrete sample dataset, find match model output

    note: only one grid mask is loaded so all model variables must be on same grid; defaults to tmask;
        call multiple times for different grids (eg U,W)

    :arg data: pandas dataframe containing data to compare to. Must include the following:
        'dtUTC': column with UTC date and time
        'Lat': decimal latitude
        'Lon': decimal longitude
        'Z': depth, positive     NOT  required if method='ferry' or sdim=2
    :type :py:class:`pandas.DataFrame`

    :arg dict filemap: dictionary mapping names of model variables to filetypes containing them

    :arg dict fdict: dictionary mapping filetypes to their time resolution in hours

    :arg mod_start: first date of time range to match
    :type :py:class:`datetime.datetime`

    :arg mod_end: end of date range to match (not included)
    :type :py:class:`datetime.datetime`

    :arg str mod_nam_fmt: naming format for model files. options are 'nowcast' or 'long'
        'nowcast' example: 05may15/SalishSea_1h_20150505_20150505_ptrc_T.nc
        'long' example: SalishSea_1h_20150206_20150804_ptrc_T_20150427-20150506.nc
                    'long' will recursively search subdirectories (to match Vicky's storage style)

    :arg str mod_basedir: path to search for model files; defaults to nowcast-green

    :arg int mod_flen: length of model files in days; defaults to 1, which is how nowcast data is stored

    :arg str method: method to use for matching. options are:

        'bin'- return model value from grid/time interval containing observation
        'vvlBin' - same as 'bin' but consider tidal change in vertical grid
        'vvlZ' - consider tidal change in vertical grid and interpolate in the vertical
        'ferry' - match observations to top model layer
        'vertNet' - match observations to mean over a vertical range defined by
                    Z_upper and Z_lower; first try will include entire cell containing end points
                    and use e3t_0 rather than time-varying e3t

    :arg str meshPath: path to mesh file; defaults to None, in which case set to:
            '/results/forcing/atmospheric/GEM2.5/operational/ops_y2015m01d01.nc' if maskName is 'ops'
            '/ocean/eolson/MEOPAR/NEMO-forcing/grid/mesh_mask201702_noLPE.nc' else (SalishSeaCast)

    :arg str maskName: variable name for mask in mesh file (check code for consistency if not tmask)
                       for ops vars use 'ops'

    :arg boolean wrapSearch: if True, use wrapper on find_closest_model_point that assumes
                             nearness of subsequent values

    :arg int wrapTol: assumed search radius from previous grid point if wrapSearch=True

    :arg str e3tvar: name of tgrid thicknesses variable; only for method=interpZe3t, which only works on t grid

    :arg Dataset fid: optionally include name of a single dataset when looping is not necessary and all matches come from
        a single file

    :arg int sdim: optionally enter number of spatial dimensions (must be the same for all variables per call);
        defaults to 3; use to match to 2d fields like ssh

    :arg boolean quiet: if True, suppress non-critical warnings

    :arg boolean preIndexed: set True if horizontal  grid indices already in input dataframe; for
               speed; not implemented with all options
    """
    # define dictionaries of mesh lat and lon variables to use with different grids:
    lonvar={'tmask':'nav_lon','umask':'glamu','vmask':'glamv','fmask':'glamf'}
    latvar={'tmask':'nav_lat','umask':'gphiu','vmask':'gphiv','fmask':'gphif'}

    # check that required columns are in dataframe:
    if method == 'ferry' or sdim==2:
        reqsubset=['dtUTC','Lat','Lon']
        if preIndexed:
            reqsubset=['dtUTC','i','j']
    elif method == 'vertNet':
        reqsubset=['dtUTC','Lat','Lon','Z_upper','Z_lower']
        if preIndexed:
            reqsubset=['dtUTC','i','j','Z_upper','Z_lower']
    else:
        reqsubset=['dtUTC','Lat','Lon','Z']
        if preIndexed:
            reqsubset=['dtUTC','i','j','k']
    if not set(reqsubset) <= set(data.keys()):

        raise Exception('{} missing from data'.format([el for el in set(reqsubset)-set(data.keys())],'%s'))

    fkeysVar=list(filemap.keys()) # list of model variables to return
    # don't load more files than necessary:
    ftypes=list(fdict.keys())
    for ikey in ftypes:
        if ikey not in set(filemap.values()):
            fdict.pop(ikey)
    if len(set(filemap.values())-set(fdict.keys()))>0:
        print('Error: file(s) missing from fdict:',set(filemap.values())-set(fdict.keys()))
    ftypes=list(fdict.keys()) # list of filetypes to containing the desired model variables
    # create inverted version of filemap dict mapping file types to the variables they contain
    filemap_r=dict()
    for ift in ftypes:
        filemap_r[ift]=list()
    for ikey in filemap:
        filemap_r[filemap[ikey]].append(ikey)

    # if mod_start and mod_end not provided, use min and max of data datetimes
    if mod_start is None:
        mod_start=np.min(data['dtUTC'])
        print(mod_start)
    if mod_end is None:
        mod_end=np.max(data['dtUTC'])
        print(mod_end)
    # adjustments to data dataframe to avoid unnecessary calculations
    data=data.loc[(data.dtUTC>=mod_start)&(data.dtUTC<mod_end)].copy(deep=True)
    data=data.dropna(how='any',subset=reqsubset) #.dropna(how='all',subset=[*varmap.keys()])
    if maskname=='ops':
        # set default mesh file for ops data (atmos forcing)
        if meshPath==None:
            meshPath='/results/forcing/atmospheric/GEM2.5/operational/ops_y2015m01d01.nc'
        # load lat, lon, and mask (all ones for ops - no land in sky)
        with nc.Dataset(meshPath) as fmesh:
            navlon=np.squeeze(np.copy(fmesh.variables['nav_lon'][:,:]-360))
            navlat=np.squeeze(np.copy(fmesh.variables['nav_lat'][:,:]))
        omask=np.expand_dims(np.ones(np.shape(navlon)),axis=(0,1))
        nemops='GEM2.5'
    else:
        # set default mesh file for SalishSeaCast data
        if meshPath==None:
            meshPath='/ocean/eolson/MEOPAR/NEMO-forcing/grid/mesh_mask201702_noLPE.nc'
        # load lat lon and ocean mask
        with nc.Dataset(meshPath) as fmesh:
            omask=np.copy(fmesh.variables[maskname])
            navlon=np.squeeze(np.copy(fmesh.variables[lonvar[maskname]][:,:]))
            navlat=np.squeeze(np.copy(fmesh.variables[latvar[maskname]][:,:]))
            if method == 'vertNet':
                e3t0=np.squeeze(np.copy(fmesh.variables['e3t_0'][0,:,:,:]))
                if maskname != 'tmask':
                    print('Warning: Using tmask thickness for variable on different grid')
        nemops='NEMO'

    # handle horizontal gridding as necessary; make sure data is in order of ascending time
    if not preIndexed:
        # find location of each obs on model grid and add to data as additional columns 'i' and 'j'
        data=et._gridHoriz(data,omask,navlon,navlat,wrapSearch,wrapTol,fastSearch, quiet=quiet,nemops=nemops)
        data=data.sort_values(by=[ix for ix in ['dtUTC','Z','j','i'] if ix in reqsubset]) # preserve list order
    else:
        data=data.sort_values(by=[ix for ix in ['dtUTC','k','j','i'] if ix in reqsubset]) # preserve list order
    data.reset_index(drop=True,inplace=True)

    # set up columns to accept model values; prepend 'mod' to distinguish from obs names
    for ivar in filemap.keys():
        data['mod_'+ivar]=np.full(len(data),np.nan)

    # create dictionary of dataframes of filename, start time, and end time for each file type
    flist=dict()
    for ift in ftypes:
        flist[ift]=et.index_model_files(mod_start,mod_end,mod_basedir,mod_nam_fmt,mod_flen,ift,fdict[ift])

    # call a function to carry out vertical matching based on specified method
    if method == 'bin':
        data = _binmatch(data,flist,ftypes,filemap_r,omask,maskP,sdim,preIndexed=preIndexed)
    elif method == 'ferry':
        print('data is matched to shallowest model level')
        data = _ferrymatch(data,flist,ftypes,filemap_r,omask,fdict)
    elif method == 'vvlZ':
        data = _interpvvlZ(data,flist,ftypes,filemap,filemap_r,omask,fdict,e3tvar)
    elif method == 'vvlBin':
        data= _vvlBin(data,flist,ftypes,filemap,filemap_r,omask,fdict,e3tvar)
    elif method == 'vertNet':
        data = _vertNetmatch(data,flist,ftypes,filemap_r,omask,e3t0,maskP)
    elif method == 'salinity':
        print('Matching by salinity...')
        data = _salinityMatch(data,flist,ftypes,filemap_r,omask,fdict)
    else:
        print('option '+method+' not written yet')
        return
    data.reset_index(drop=True,inplace=True)
    return data

In [ ]:
def _salinityMatch(data, flist, ftypes, filemap_r, omask, fdict):
    import xarray as xr
    import numpy as np
    from tqdm import tqdm

    #varname = 'vosaline'
    matched_salinities = []

    # Find which filetype contains salinity
    salinity_var, salinity_ftype = None, None
    for ftype in ftypes:
        for var in filemap_r[ftype]:
            if 'sal' in var.lower():  ## was just 'sal' before
                salinity_var = var
                salinity_ftype = ftype
                break
        if salinity_var:
            break
    if not salinity_var:
        raise ValueError("No salinity variable found in filemap_r.")

    salinity_files = flist[salinity_ftype]
    salinity_files.columns = ['fname', 'start', 'end']

    # Cache for xarray datasets
    dataset_cache = {}

    for idx, row in tqdm(data.iterrows(), total=len(data)):
        obs_time = row['dtUTC']
        obs_sal = row['SA']
        j, i = int(row['j']), int(row['i'])
        k = None

        # Step 1: Find matching salinity depth
        for _, mf in salinity_files.iterrows():
            if mf['start'] <= obs_time < mf['end']:
                fname = mf['fname']
                if fname not in dataset_cache:
                    dataset_cache[fname] = xr.open_dataset(fname)
                ds = dataset_cache[fname]

                try:
                    # Select time (nearest if needed), then slice j,i
                    sel = ds[salinity_var].sel(time_counter=obs_time, method='nearest')
                    sal_profile = sel[:, j, i].values  # depth profile

                    if np.isnan(sal_profile).all():
                        matched_salinities.append(np.nan)
                        k = None
                    else:
                        sal_diff = np.abs(sal_profile - obs_sal)
                        k = np.nanargmin(sal_diff)
                        matched_salinities.append(sal_profile[k])
                except Exception as e:
                    print(f"Error reading salinity at {fname}: {e}")
                    matched_salinities.append(np.nan)
                    k = None
                break

        if k is None:
            # Fill all variables with NaN
            for ft in ftypes:
                for var in filemap_r[ft]:
                    data.at[idx, f'mod_{var}'] = np.nan
            continue

        # Step 2: Grab each variable at (time, k, j, i) using xarray
        for ft in ftypes:
            var_files = flist[ft]
            var_files.columns = ['fname', 'start', 'end']

            for _, mf in var_files.iterrows():
                if mf['start'] <= obs_time < mf['end']:
                    fname = mf['fname']
                    if fname not in dataset_cache:
                        dataset_cache[fname] = xr.open_dataset(fname)
                    ds = dataset_cache[fname]

                    for var in filemap_r[ft]:
                        try:
                            sel = ds[var].sel(time_counter=obs_time, method='nearest')
                            val = sel[k, j, i].item()
                        except Exception:
                            val = np.nan
                        data.at[idx, f'mod_{var}'] = val
                    break

    # Close all xarray datasets
    for ds in dataset_cache.values():
        ds.close()

    data['matched_salinity'] = matched_salinities
    return data

: 

In [ ]:
matched_data15 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2015,1,1),
    mod_end=dt.datetime(2015,12,31),
    method='salinity')
matched_data15.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20150101_20151231.csv') 

(Lat,Lon)= 25.32 -81.17  not matched to domain
(Lat,Lon)= 25.34 -81.16  not matched to domain
(Lat,Lon)= 25.34 -81.15  not matched to domain
(Lat,Lon)= 25.35 -81.65  not matched to domain
(Lat,Lon)= 25.35 -81.5  not matched to domain
(Lat,Lon)= 25.35 -81.49  not matched to domain
(Lat,Lon)= 25.35 -81.34  not matched to domain
(Lat,Lon)= 25.35 -81.27  not matched to domain
(Lat,Lon)= 25.35 -81.23  not matched to domain
(Lat,Lon)= 25.35 -81.2  not matched to domain
(Lat,Lon)= 25.35 -81.15  not matched to domain
(Lat,Lon)= 26.98 -79.5  not matched to domain
(Lat,Lon)= 26.98 -79.27  not matched to domain
(Lat,Lon)= 26.98 -79.18  not matched to domain
(Lat,Lon)= 26.99 -79.87  not matched to domain
(Lat,Lon)= 26.99 -79.78  not matched to domain
(Lat,Lon)= 26.99 -79.5  not matched to domain
(Lat,Lon)= 26.99 -79.27  not matched to domain
(Lat,Lon)= 26.99 -79.18  not matched to domain
(Lat,Lon)= 27.0 -80.0  not matched to domain
(Lat,Lon)= 27.0 -79.93  not matched to domain
(Lat,Lon)= 27.0 -79.

  3%|▎         | 807/31414 [01:28<13:01, 39.16it/s]   

Error reading salinity at /results2/SalishSea/nowcast-green.202111/20jan15/SalishSea_1h_20150120_20150120_grid_T.nc: All-NaN slice encountered


  3%|▎         | 998/31414 [01:33<12:28, 40.64it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/20jan15/SalishSea_1h_20150120_20150120_grid_T.nc: All-NaN slice encountered


...

  7%|▋         | 2088/31414 [02:03<12:06, 40.34it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/21jan15/SalishSea_1h_20150121_20150121_grid_T.nc: All-NaN slice encountered


  7%|▋         | 2191/31414 [02:10<13:10, 36.98it/s]  

Error reading salinity at /results2/SalishSea/nowcast-green.202111/22jan15/SalishSea_1h_20150122_20150122_grid_T.nc: All-NaN slice encountered


  8%|▊         | 2573/31414 [03:30<26:01:17,  3.25s/it]

In [ ]:
matched_data16 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2016,1,1),
    mod_end=dt.datetime(2016,12,31),
    method='salinity')
matched_data16.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20160101_20161231.csv') 

(Lat,Lon)= 25.59 -113.78  not matched to domain
(Lat,Lon)= 25.75 -113.46  not matched to domain
(Lat,Lon)= 25.92 -113.14  not matched to domain
(Lat,Lon)= 26.09 -112.82  not matched to domain
(Lat,Lon)= 26.18 -112.63  not matched to domain
(Lat,Lon)= 27.39 -116.19  not matched to domain
(Lat,Lon)= 27.55 -115.87  not matched to domain
(Lat,Lon)= 27.72 -115.54  not matched to domain
(Lat,Lon)= 27.79 -115.41  not matched to domain
(Lat,Lon)= 27.85 -115.29  not matched to domain
(Lat,Lon)= 28.05 -114.89  not matched to domain
(Lat,Lon)= 28.22 -114.57  not matched to domain
(Lat,Lon)= 28.39 -114.23  not matched to domain
(Lat,Lon)= 28.44 -114.14  not matched to domain
(Lat,Lon)= 30.69 -118.79  not matched to domain
(Lat,Lon)= 30.85 -118.45  not matched to domain
(Lat,Lon)= 31.02 -118.12  not matched to domain
(Lat,Lon)= 31.19 -117.79  not matched to domain
(Lat,Lon)= 31.45 -117.45  not matched to domain
(Lat,Lon)= 31.52 -117.11  not matched to domain
(Lat,Lon)= 31.62 -116.91  not matched to

  2%|▏         | 525/34275 [01:28<2:34:48,  3.63it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/14jan16/SalishSea_1h_20160114_20160114_grid_T.nc: All-NaN slice encountered


  3%|▎         | 1097/34275 [01:41<51:01, 10.84it/s] 


KeyboardInterrupt: 

In [ ]:
matched_data17 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2017,1,1),
    mod_end=dt.datetime(2017,12,31),
    method='salinity')
matched_data17.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20170101_20171231.csv') 

(Lat,Lon)= 18.83 -93.07  not matched to domain
(Lat,Lon)= 19.17 -93.3  not matched to domain
(Lat,Lon)= 19.61 -93.51  not matched to domain
(Lat,Lon)= 20.02 -93.78  not matched to domain
(Lat,Lon)= 20.6 -94.29  not matched to domain
(Lat,Lon)= 20.73 -94.73  not matched to domain
(Lat,Lon)= 21.17 -90.76  not matched to domain
(Lat,Lon)= 21.45 -91.57  not matched to domain
(Lat,Lon)= 21.55 -86.75  not matched to domain
(Lat,Lon)= 21.59 -86.5  not matched to domain
(Lat,Lon)= 21.63 -86.25  not matched to domain
(Lat,Lon)= 21.67 -86.01  not matched to domain
(Lat,Lon)= 21.72 -85.72  not matched to domain
(Lat,Lon)= 21.74 -92.31  not matched to domain
(Lat,Lon)= 21.76 -85.47  not matched to domain
(Lat,Lon)= 21.8 -85.21  not matched to domain
(Lat,Lon)= 21.83 -84.98  not matched to domain
(Lat,Lon)= 21.93 -88.0  not matched to domain
(Lat,Lon)= 22.0 -92.9  not matched to domain
(Lat,Lon)= 22.27 -97.73  not matched to domain
(Lat,Lon)= 22.27 -97.65  not matched to domain
(Lat,Lon)= 22.27 -97

  1%|          | 343/32439 [00:48<10:49, 49.44it/s] 

Error reading salinity at /results2/SalishSea/nowcast-green.202111/11jan17/SalishSea_1h_20170111_20170111_grid_T.nc: All-NaN slice encountered


  1%|▏         | 439/32439 [00:50<10:36, 50.27it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/11jan17/SalishSea_1h_20170111_20170111_grid_T.nc: All-NaN slice encountered


...

Error reading salinity at /results2/SalishSea/nowcast-green.202111/11jan17/SalishSea_1h_20170111_20170111_grid_T.nc: All-NaN slice encountered


  2%|▏         | 534/32439 [00:52<11:06, 47.84it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/11jan17/SalishSea_1h_20170111_20170111_grid_T.nc: All-NaN slice encountered


  2%|▏         | 626/32439 [00:57<49:05, 10.80it/s]  


KeyboardInterrupt: 

In [ ]:
matched_data18 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2018,1,1),
    mod_end=dt.datetime(2018,12,31),
    method='salinity')
matched_data18.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20180101_20181231.csv') 

(Lat,Lon)= 26.94 -79.62  not matched to domain
(Lat,Lon)= 26.95 -79.62  not matched to domain
(Lat,Lon)= 26.96 -79.62  not matched to domain
(Lat,Lon)= 26.99 -79.92  not matched to domain
(Lat,Lon)= 26.99 -79.78  not matched to domain
(Lat,Lon)= 26.99 -79.63  not matched to domain
(Lat,Lon)= 26.99 -79.18  not matched to domain
(Lat,Lon)= 27.0 -79.92  not matched to domain
(Lat,Lon)= 27.0 -79.86  not matched to domain
(Lat,Lon)= 27.0 -79.78  not matched to domain
(Lat,Lon)= 27.0 -79.63  not matched to domain
(Lat,Lon)= 27.01 -79.98  not matched to domain
(Lat,Lon)= 27.01 -79.86  not matched to domain
(Lat,Lon)= 27.01 -79.78  not matched to domain
(Lat,Lon)= 27.01 -79.63  not matched to domain
(Lat,Lon)= 27.02 -79.86  not matched to domain
(Lat,Lon)= 28.75 -80.57  not matched to domain
(Lat,Lon)= 28.77 -80.43  not matched to domain
(Lat,Lon)= 28.78 -80.43  not matched to domain
(Lat,Lon)= 28.82 -80.13  not matched to domain
(Lat,Lon)= 28.86 -79.98  not matched to domain
(Lat,Lon)= 28.87 

  0%|          | 88/31524 [01:22<20:44, 25.25it/s]   

Error reading salinity at /results2/SalishSea/nowcast-green.202111/10jan18/SalishSea_1h_20180110_20180110_grid_T.nc: All-NaN slice encountered


  0%|          | 130/31524 [01:23<11:15, 46.45it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/10jan18/SalishSea_1h_20180110_20180110_grid_T.nc: All-NaN slice encountered
Error reading salinity at /results2/SalishSea/nowcast-green.202111/10jan18/SalishSea_1h_20180110_20180110_grid_T.nc: All-NaN slice encountered


  1%|          | 279/31524 [01:26<10:06, 51.52it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/10jan18/SalishSea_1h_20180110_20180110_grid_T.nc: All-NaN slice encountered


  1%|          | 331/31524 [01:33<2:27:21,  3.53it/s]


KeyboardInterrupt: 

In [ ]:
matched_data19 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2019,1,1),
    mod_end=dt.datetime(2019,12,31),
    method='salinity')
matched_data19.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20190101_20191231.csv') 

(Lat,Lon)= 44.36 -124.94  not matched to domain
(Lat,Lon)= 44.37 -124.96  not matched to domain
(Lat,Lon)= 44.37 -124.95  not matched to domain
(Lat,Lon)= 44.38 -124.95  not matched to domain
(Lat,Lon)= 44.52 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.38  not matched to domain
(Lat,Lon)= 44.64 -124.31  not matched to domain
(Lat,Lon)= 44.65 -128.77  not matched to domain
(Lat,Lon)= 44.65 -128.18  not matched to domain
(Lat,Lon)= 44.65 -127.59  not matched to domain
(Lat,Lon)= 44.65 -127.0  not matched to domain
(Lat,Lon)= 44.65 -126.55  not matched to domain
(Lat,Lon)= 44.65 -126.05  not matched to domain
(Lat,Lon)= 44.65 -125.6  not matched to domain
(Lat,Lon)= 44.65 -125.12  not matched to domain
(Lat,Lon)= 44.65 -124.88  not matched to domain
(Lat,Lon)= 44.65 -124.65  not matched to domain
(Lat,Lon)= 44.65 -124.53  not matched to domain
(Lat,Lon)= 44.65 -124.52  not matched to domain
(Lat,Lon)= 44.65 -124.41  not matched to d

  0%|          | 129/30725 [01:02<10:44, 47.44it/s]  

Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan19/SalishSea_1h_20190107_20190107_grid_T.nc: All-NaN slice encountered


  1%|          | 268/30725 [01:06<11:13, 45.21it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan19/SalishSea_1h_20190107_20190107_grid_T.nc: All-NaN slice encountered


...

Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan19/SalishSea_1h_20190107_20190107_grid_T.nc: All-NaN slice encountered


  3%|▎         | 829/30725 [01:22<10:06, 49.31it/s]  

Error reading salinity at /results2/SalishSea/nowcast-green.202111/08jan19/SalishSea_1h_20190108_20190108_grid_T.nc: All-NaN slice encountered


  6%|▋         | 1949/30725 [02:22<35:03, 13.68it/s]  


KeyboardInterrupt: 

In [ ]:
matched_data20 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2020,1,1),
    mod_end=dt.datetime(2020,12,31),
    method='salinity')
matched_data20.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20200101_20201231.csv') 

(Lat,Lon)= 44.36 -124.94  not matched to domain
(Lat,Lon)= 44.37 -124.96  not matched to domain
(Lat,Lon)= 44.37 -124.95  not matched to domain
(Lat,Lon)= 44.38 -124.95  not matched to domain
(Lat,Lon)= 44.52 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.38  not matched to domain
(Lat,Lon)= 44.64 -124.31  not matched to domain
(Lat,Lon)= 44.65 -128.77  not matched to domain
(Lat,Lon)= 44.65 -128.18  not matched to domain
(Lat,Lon)= 44.65 -127.59  not matched to domain
(Lat,Lon)= 44.65 -127.0  not matched to domain
(Lat,Lon)= 44.65 -126.55  not matched to domain
(Lat,Lon)= 44.65 -126.05  not matched to domain
(Lat,Lon)= 44.65 -125.6  not matched to domain
(Lat,Lon)= 44.65 -125.12  not matched to domain
(Lat,Lon)= 44.65 -124.88  not matched to domain
(Lat,Lon)= 44.65 -124.65  not matched to domain
(Lat,Lon)= 44.65 -124.53  not matched to domain
(Lat,Lon)= 44.65 -124.52  not matched to domain
(Lat,Lon)= 44.65 -124.41  not matched to d

  9%|▉         | 1170/13154 [02:31<04:13, 47.21it/s] 

Error reading salinity at /results2/SalishSea/nowcast-green.202111/17jan20/SalishSea_1h_20200117_20200117_grid_T.nc: All-NaN slice encountered


 22%|██▏       | 2863/13154 [07:25<51:54,  3.30it/s]  

Error reading salinity at /results2/SalishSea/nowcast-green.202111/19feb20/SalishSea_1h_20200219_20200219_grid_T.nc: All-NaN slice encountered


...

 86%|████████▌ | 11322/13154 [52:09<01:54, 15.98it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/09dec20/SalishSea_1h_20201209_20201209_grid_T.nc: All-NaN slice encountered


 96%|█████████▌| 12633/13154 [55:08<07:29,  1.16it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/22dec20/SalishSea_1h_20201222_20201222_grid_T.nc: All-NaN slice encountered
Error reading salinity at /results2/SalishSea/nowcast-green.202111/22dec20/SalishSea_1h_20201222_20201222_grid_T.nc: All-NaN slice encountered
Error reading salinity at /results2/SalishSea/nowcast-green.202111/22dec20/SalishSea_1h_20201222_20201222_grid_T.nc: All-NaN slice encountered


100%|██████████| 13154/13154 [56:39<00:00,  3.87it/s]


In [ ]:
matched_data21 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2021,1,1),
    mod_end=dt.datetime(2021,12,31),
    method='salinity')
matched_data21.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20210101_20211231.csv') 

(Lat,Lon)= 44.36 -124.94  not matched to domain
(Lat,Lon)= 44.37 -124.96  not matched to domain
(Lat,Lon)= 44.37 -124.95  not matched to domain
(Lat,Lon)= 44.38 -124.95  not matched to domain
(Lat,Lon)= 44.52 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.38  not matched to domain
(Lat,Lon)= 44.64 -124.31  not matched to domain
(Lat,Lon)= 44.65 -128.77  not matched to domain
(Lat,Lon)= 44.65 -128.18  not matched to domain
(Lat,Lon)= 44.65 -127.0  not matched to domain
(Lat,Lon)= 44.65 -126.55  not matched to domain
(Lat,Lon)= 44.65 -126.05  not matched to domain
(Lat,Lon)= 44.65 -125.6  not matched to domain
(Lat,Lon)= 44.65 -125.12  not matched to domain
(Lat,Lon)= 44.65 -124.88  not matched to domain
(Lat,Lon)= 44.65 -124.65  not matched to domain
(Lat,Lon)= 44.65 -124.53  not matched to domain
(Lat,Lon)= 44.65 -124.52  not matched to domain
(Lat,Lon)= 44.65 -124.41  not matched to domain
(Lat,Lon)= 44.65 -124.29  not matched to d

  0%|          | 45/32916 [00:20<3:37:12,  2.52it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan21/SalishSea_1h_20210107_20210107_grid_T.nc: All-NaN slice encountered


  0%|          | 61/32916 [00:24<2:00:44,  4.54it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan21/SalishSea_1h_20210107_20210107_grid_T.nc: All-NaN slice encountered
Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan21/SalishSea_1h_20210107_20210107_grid_T.nc: All-NaN slice encountered
Error reading salinity at /results2/SalishSea/nowcast-green.202111/07jan21/SalishSea_1h_20210107_20210107_grid_T.nc: All-NaN slice encountered


...

Error reading salinity at /results2/SalishSea/nowcast-green.202111/16dec21/SalishSea_1h_20211216_20211216_grid_T.nc: All-NaN slice encountered


 99%|█████████▊| 32485/32916 [1:01:38<06:14,  1.15it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/21dec21/SalishSea_1h_20211221_20211221_grid_T.nc: All-NaN slice encountered
Error reading salinity at /results2/SalishSea/nowcast-green.202111/21dec21/SalishSea_1h_20211221_20211221_grid_T.nc: All-NaN slice encountered


100%|██████████| 32916/32916 [1:03:01<00:00,  8.70it/s]


: 

In [ ]:
matched_data22 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2022,1,1),
    mod_end=dt.datetime(2022,12,31),
    method='salinity')
matched_data22.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20220101_20221231.csv') 

(Lat,Lon)= 44.36 -124.94  not matched to domain
(Lat,Lon)= 44.37 -124.96  not matched to domain
(Lat,Lon)= 44.37 -124.95  not matched to domain
(Lat,Lon)= 44.38 -124.95  not matched to domain
(Lat,Lon)= 44.52 -125.39  not matched to domain
(Lat,Lon)= 44.53 -125.39  not matched to domain
(Lat,Lon)= 44.64 -124.31  not matched to domain
(Lat,Lon)= 44.66 -124.1  not matched to domain
(Lat,Lon)= 45.82 -129.75  not matched to domain
(Lat,Lon)= 45.93 -130.01  not matched to domain
(Lat,Lon)= 45.94 -129.97  not matched to domain
(Lat,Lon)= 46.02 -130.44  not matched to domain
(Lat,Lon)= 46.07 -130.44  not matched to domain
(Lat,Lon)= 46.07 -130.39  not matched to domain
(Lat,Lon)= 46.09 -144.01  not matched to domain
(Lat,Lon)= 46.45 -124.01  not matched to domain
(Lat,Lon)= 46.46 -123.94  not matched to domain
(Lat,Lon)= 46.54 -123.98  not matched to domain
(Lat,Lon)= 46.64 -123.99  not matched to domain
(Lat,Lon)= 46.66 -131.17  not matched to domain
(Lat,Lon)= 46.69 -131.41  not matched to 

  0%|          | 48/30628 [00:09<1:05:37,  7.77it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/04jan22/SalishSea_1h_20220104_20220104_grid_T.nc: All-NaN slice encountered


  0%|          | 58/30628 [00:12<1:30:45,  5.61it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/04jan22/SalishSea_1h_20220104_20220104_grid_T.nc: All-NaN slice encountered


...

 34%|███▍      | 10479/30628 [13:31<1:51:32,  3.01it/s]

Error reading salinity at /results2/SalishSea/nowcast-green.202111/07apr22/SalishSea_1h_20220407_20220407_grid_T.nc: All-NaN slice encountered


 39%|███▉      | 12044/30628 [18:23<1:24:57,  3.65it/s] 

Error reading salinity at /results2/SalishSea/nowcast-green.202111/10may22/SalishSea_1h_20220510_20220510_grid_T.nc: All-NaN slice encountered


 40%|███▉      | 12247/30628 [18:38<12:22, 24.75it/s]  

In [ ]:
matched_data23 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2023,1,1),
    mod_end=dt.datetime(2023,12,31),
    method='salinity')
matched_data23.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20230101_20231231.csv') 

In [ ]:
matched_data24 = matchData(
    data=data,  # your observations DataFrame
    filemap={'nitrate':'biol_T','silicon':'biol_T','ammonium':'biol_T','diatoms':'biol_T','flagellates':'biol_T',
    'vosaline':'grid_T','votemper':'grid_T',
    'total_alkalinity':'chem_T','dissolved_inorganic_carbon':'chem_T','dissolved_oxygen':'chem_T',
        },  # tell it which model variable to match
    fdict={'biol_T':1,'grid_T':1,'chem_T':1},  # model file timestep in hours
    mod_start=dt.datetime(2024,1,1),
    mod_end=dt.datetime(2024,12,31),
    method='salinity')
matched_data24.to_csv('/ocean/atall/MOAD/ObsModel/202111/ObsModel_202111_pnwBySalinity_20240101_20241231.csv') 